# 03 - Model Selection and Evaluation

## Objetivo

Establecer un baseline y comparar varias familias de modelos de
regresión mediante validación cruzada.

La métrica principal será RMSLE, la utilizada por la competición
House Prices de Kaggle. Utilizaremos MAE y RMSE como métricas
complementarias para interpretar el tamaño de los errores.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Localizar la raíz del proyecto tanto si ejecutamos el notebook
# desde la raíz como desde la carpeta notebooks.
candidates = [Path.cwd(), Path.cwd().parent]

ROOT_DIR = next(
    path for path in candidates
    if (path / "data" / "train.csv").exists()
    and (path / "src").exists()
)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing import build_pipeline


# Cargar datos.
train = pd.read_csv(ROOT_DIR / "data" / "train.csv")

X = train.drop(columns=["SalePrice", "Id"])
y = train["SalePrice"].copy()

# Reproducir exactamente la separación de la fase anterior.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Entrenamiento:", X_train.shape)
print("Validación reservada:", X_valid.shape)

Entrenamiento: (1168, 79)
Validación reservada: (292, 79)


In [7]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

def log_target_model(estimator, scale_numeric=False):
    """Entrena el estimador sobre log1p(SalePrice)
    y devuelve predicciones en dólares.
    """
    pipeline = build_pipeline(
        X_train,
        estimator,
        scale_numeric=scale_numeric,
    )

    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
    )


models = {
    "Baseline (media log)": log_target_model(
        DummyRegressor(strategy="mean")
    ),

    "Ridge (precio original)": build_pipeline(
        X_train,
        Ridge(alpha=10.0),
        scale_numeric=True,
    ),

    "Ridge (objetivo log)": log_target_model(
        Ridge(alpha=10.0),
        scale_numeric=True,
    ),

    "Random Forest (log)": log_target_model(
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        )
    ),

    "Extra Trees (log)": log_target_model(
        ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        )
    ),

    "Gradient Boosting (log)": log_target_model(
        GradientBoostingRegressor(
            random_state=42,
        )
    ),

    "Linear Regression (log)": log_target_model(
        LinearRegression(),
        scale_numeric=True,
    ),

    "SVR RBF (log)": log_target_model(
        SVR(kernel="rbf"),
        scale_numeric=True,
    ),

    "KNN (log)": log_target_model(
        KNeighborsRegressor(
            n_neighbors=5,
            weights="distance",
        ),
        scale_numeric=True,
    ),
}

print("Modelos preparados:", len(models))

Modelos preparados: 9


In [8]:
from sklearn.metrics import (
    make_scorer,
    mean_squared_log_error,
)
from sklearn.model_selection import KFold, cross_validate


def rmsle_clipped(y_true, y_pred):
    """RMSLE con predicciones negativas limitadas a cero."""
    y_pred = np.asarray(y_pred).reshape(-1)
    y_pred = np.clip(y_pred, 0, None)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )


scoring = {
    "RMSLE": make_scorer(
        rmsle_clipped,
        greater_is_better=False,
    ),
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [9]:
results = []

for name, model in models.items():
    print(f"Evaluando: {name}")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise",
    )

    results.append({
        "model": name,
        "cv_RMSLE_mean": -scores["test_RMSLE"].mean(),
        "cv_RMSLE_std": scores["test_RMSLE"].std(),
        "train_RMSLE_mean": -scores["train_RMSLE"].mean(),
        "cv_MAE_mean": -scores["test_MAE"].mean(),
        "cv_RMSE_mean": -scores["test_RMSE"].mean(),
        "fit_time_sec": scores["fit_time"].mean(),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("cv_RMSLE_mean")
    .reset_index(drop=True)
)

display(results_df.round(4))

Evaluando: Baseline (media log)
Evaluando: Ridge (precio original)


c:\Users\afern\.conda\envs\principal\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.00023838586026473272.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\afern\.conda\envs\principal\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.000237848952703955.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\afern\.conda\envs\principal\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0002433692535904337.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\afern\.conda\envs\principal\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.00023838586026473272.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T

Evaluando: Ridge (objetivo log)
Evaluando: Random Forest (log)
Evaluando: Extra Trees (log)
Evaluando: Gradient Boosting (log)
Evaluando: Linear Regression (log)
Evaluando: SVR RBF (log)
Evaluando: KNN (log)


,model,cv_RMSLE_mean,cv_RMSLE_std,train_RMSLE_mean,cv_MAE_mean,cv_RMSE_mean,fit_time_sec
0,Gradient Boosting (log),0.1330,0.0151,0.0721,16580.4480,28580.4221,0.4786
1,SVR RBF (log),0.1408,0.0240,0.0773,17051.6074,32167.1503,0.0511
2,Ridge (objetivo log),0.1412,0.0290,0.1042,17232.6440,40484.5206,0.0324
3,Random Forest (log),0.1476,0.0160,0.0548,18414.2363,31068.1352,0.3711
4,Extra Trees (log),0.1499,0.0116,0.0000,18742.1581,31675.7316,0.4815
5,Ridge (precio original),0.1589,0.0138,0.1205,18402.0924,31888.2787,0.0425
6,Linear Regression (log),0.1698,0.0262,0.0839,18941.7110,54277.3358,0.0420
7,KNN (log),0.1732,0.0130,0.0000,22294.8514,37733.7692,0.0321
8,Baseline (media log),0.3901,0.0182,0.3904,54615.5327,78327.6768,0.0324


## Optimización de hiperparámetros

In [10]:
from scipy.stats import randint, loguniform

from sklearn.model_selection import RandomizedSearchCV


# Espacio de búsqueda para cada candidato.
search_spaces = {
    "Gradient Boosting (log)": {
        "regressor__model__n_estimators": randint(100, 501),
        "regressor__model__learning_rate": loguniform(0.015, 0.15),
        "regressor__model__max_depth": [1, 2, 3, 4],
        "regressor__model__min_samples_leaf": randint(1, 16),
        "regressor__model__subsample": [0.7, 0.85, 1.0],
    },

    "SVR RBF (log)": {
        "regressor__model__C": loguniform(0.1, 1000),
        "regressor__model__epsilon": loguniform(0.001, 0.2),
        "regressor__model__gamma": [
            "scale", "auto", 0.001, 0.003, 0.01, 0.03, 0.1
        ],
    },

    "Ridge (objetivo log)": {
        "regressor__model__alpha": loguniform(0.01, 1000),
    },

    "Random Forest (log)": {
        "regressor__model__n_estimators": randint(200, 601),
        "regressor__model__max_depth": [None, 8, 12, 16, 20, 30],
        "regressor__model__min_samples_split": randint(2, 16),
        "regressor__model__min_samples_leaf": randint(1, 10),
        "regressor__model__max_features": [
            "sqrt", 0.5, 0.75, 1.0
        ],
    },

    "Extra Trees (log)": {
        "regressor__model__n_estimators": randint(200, 601),
        "regressor__model__max_depth": [None, 8, 12, 16, 20, 30],
        "regressor__model__min_samples_split": randint(2, 16),
        "regressor__model__min_samples_leaf": randint(1, 10),
        "regressor__model__max_features": [
            "sqrt", 0.5, 0.75, 1.0
        ],
    },

    "KNN (log)": {
        "regressor__model__n_neighbors": randint(3, 51),
        "regressor__model__weights": ["uniform", "distance"],
        "regressor__model__p": [1, 2],
        "regressor__model__leaf_size": randint(15, 51),
    },
}

In [ ]:
searches = {}
tuned_results = []

for name, params in search_spaces.items():
    print(f"\nOptimizando: {name}")

    search = RandomizedSearchCV(
        estimator=models[name],
        param_distributions=params,
        n_iter=20,
        scoring=scoring["RMSLE"],
        cv=cv,
        refit=True,
        random_state=42,
        n_jobs=1,
        return_train_score=True,
        error_score="raise",
        verbose=1,
    )

    search.fit(X_train, y_train)

    searches[name] = search

    best_index = search.best_index_

    tuned_results.append({
        "model": name,
        "best_cv_RMSLE": -search.best_score_,
        "cv_RMSLE_std": search.cv_results_[
            "std_test_score"
        ][best_index],
        "train_RMSLE": -search.cv_results_[
            "mean_train_score"
        ][best_index],
        "mean_fit_time_sec": search.cv_results_[
            "mean_fit_time"
        ][best_index],
        "best_params": search.best_params_,
    })

tuned_results_df = (
    pd.DataFrame(tuned_results)
    .sort_values("best_cv_RMSLE")
    .reset_index(drop=True)
)

display(
    tuned_results_df.drop(columns="best_params").round(4)
)

In [12]:
# Mostrar los mejores hiperparámetros de los dos candidatos principales.
for name in ["SVR RBF (log)", "Gradient Boosting (log)"]:
    search = searches[name]

    print(f"\n=== {name} ===")
    print(f"RMSLE medio en CV: {-search.best_score_:.4f}")
    print("Mejores parámetros:")

    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")


=== SVR RBF (log) ===
RMSLE medio en CV: 0.1180
Mejores parámetros:
  regressor__model__C: 5.342937261279771
  regressor__model__epsilon: 0.004678719265016202
  regressor__model__gamma: 0.001

=== Gradient Boosting (log) ===
RMSLE medio en CV: 0.1248
Mejores parámetros:
  regressor__model__learning_rate: 0.022213417679962773
  regressor__model__max_depth: 3
  regressor__model__min_samples_leaf: 2
  regressor__model__n_estimators: 487
  regressor__model__subsample: 0.7


In [13]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

refined_spaces = {
    "SVR RBF (log)": {
        "regressor__model__C": loguniform(0.5, 50),
        "regressor__model__epsilon": loguniform(0.0005, 0.03),
        "regressor__model__gamma": loguniform(0.0001, 0.01),
    },
    "Gradient Boosting (log)": {
        "regressor__model__n_estimators": randint(300, 1001),
        "regressor__model__learning_rate": loguniform(0.005, 0.06),
        "regressor__model__max_depth": [2, 3, 4],
        "regressor__model__min_samples_leaf": randint(2, 10),
        "regressor__model__subsample": [0.7, 0.85, 1.0],
    },
}

refined_searches = {}
refined_results = []

for name, params in refined_spaces.items():
    print(f"\nRefinando: {name}")

    search = RandomizedSearchCV(
        estimator=models[name],
        param_distributions=params,
        n_iter=20,
        scoring=scoring["RMSLE"],
        cv=cv,
        refit=True,
        random_state=43,
        n_jobs=1,
        return_train_score=True,
        error_score="raise",
        verbose=1,
    )

    search.fit(X_train, y_train)

    refined_searches[name] = search

    idx = search.best_index_

    refined_results.append({
        "model": name,
        "best_cv_RMSLE": -search.best_score_,
        "cv_RMSLE_std": search.cv_results_[
            "std_test_score"
        ][idx],
        "train_RMSLE": -search.cv_results_[
            "mean_train_score"
        ][idx],
        "best_params": search.best_params_,
    })

refined_results_df = (
    pd.DataFrame(refined_results)
    .sort_values("best_cv_RMSLE")
    .reset_index(drop=True)
)

display(
    refined_results_df.drop(columns="best_params").round(4)
)


Refinando: SVR RBF (log)
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Refinando: Gradient Boosting (log)
Fitting 5 folds for each of 20 candidates, totalling 100 fits


,model,best_cv_RMSLE,cv_RMSLE_std,train_RMSLE
0,SVR RBF (log),0.1195,0.0196,0.0722
1,Gradient Boosting (log),0.1231,0.0169,0.0662


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Candidatos ya optimizados. No volvemos a ajustar sus parámetros.
final_candidates = {
    "SVR inicial optimizado": searches["SVR RBF (log)"],
    "SVR refinado": refined_searches["SVR RBF (log)"],
    "Gradient Boosting refinado":
        refined_searches["Gradient Boosting (log)"],
}

final_results = []
final_predictions = {}

for name, search in final_candidates.items():

    # RandomizedSearchCV con refit=True ya ha ajustado
    # la mejor configuración usando todo X_train.
    model = search.best_estimator_

    predictions = np.clip(
        model.predict(X_valid),
        0,
        None,
    )

    final_predictions[name] = predictions

    final_results.append({
        "model": name,
        "CV_RMSLE": -search.best_score_,
        "Validation_RMSLE": rmsle_clipped(
            y_valid, predictions
        ),
        "Validation_MAE": mean_absolute_error(
            y_valid, predictions
        ),
        "Validation_RMSE": np.sqrt(
            mean_squared_error(y_valid, predictions)
        ),
    })

final_results_df = (
    pd.DataFrame(final_results)
    .sort_values("CV_RMSLE")
    .reset_index(drop=True)
)

display(final_results_df.round(4))

,model,CV_RMSLE,Validation_RMSLE,Validation_MAE,Validation_RMSE
0,SVR inicial optimizado,0.1180,0.1313,14419.8373,24103.5479
1,SVR refinado,0.1195,0.1282,14515.3283,24812.1099
2,Gradient Boosting refinado,0.1231,0.1358,15885.7785,27060.3427


## Conclusiones de la selección de modelos

Se compararon nueve configuraciones iniciales mediante validación cruzada de cinco particiones, utilizando RMSLE como métrica principal.

Posteriormente se optimizaron los hiperparámetros de Ridge, SVR, Random Forest, Extra Trees, KNN y Gradient Boosting mediante `RandomizedSearchCV`.

El mejor resultado fue obtenido por **SVR con kernel RBF y objetivo logarítmico**, con un RMSLE medio de 0,1180 en validación cruzada. Sus hiperparámetros seleccionados son `C=5.34294`, `epsilon=0.004679` y `gamma=0.001`.

Gradient Boosting se conserva como modelo alternativo, con un RMSLE medio de 0,1231 después de optimizar sus hiperparámetros.

La evaluación sobre el conjunto local reservado favoreció al SVR refinado en RMSLE, aunque el SVR inicial obtuvo mejores resultados en MAE y RMSE. Dado que la validación cruzada es nuestro criterio principal, mantenemos el SVR inicial como candidato final. La diferencia entre configuraciones no permite afirmar que exista una superioridad universal.

Los resultados del conjunto reservado se interpretan como una comprobación adicional, no como una estimación completamente independiente, porque ya se consultó durante el proceso de selección.

En la siguiente fase se reentrenará el modelo seleccionado utilizando todas las viviendas etiquetadas disponibles y se generará el archivo de entrega para Kaggle.